> **About this version.** This notebook is based on Pinecone's [`00-rerankers.ipynb`](https://github.com/pinecone-io/examples/blob/master/learn/generation/better-rag/00-rerankers.ipynb) (MIT License), the companion to the chapter [Rerankers and Two-Stage Retrieval](https://www.pinecone.io/learn/series/rag/rerankers/). This version updates the models, fixes the bugs that stopped it from running, and adds a section that measures what the reranker adds: **Measuring the Reranker**, near the end. The README lists every change.

# Rerankers

Rerankers have been a common component of retrieval pipelines for many years. They allow us to add a final "reranking" step to our retrieval pipelines — like with **R**etrieval **A**ugmented **G**eneration (RAG) — that can be used to dramatically optimize our retrieval pipelines and improve their accuracy.

In the example notebook we'll learn how to create retrieval pipelines with reranking using the [Cohere reranking model](https://txt.cohere.com/rerank/) (which is available for free).

To begin, we setup our prerequisite libraries.

In [1]:
%pip install -qU \
    datasets==2.14.5 \
    openai==1.6.1 \
    pinecone-client==3.1.0 \
    cohere==4.27

Note: you may need to restart the kernel to use updated packages.


## Data Preparation

We start by downloading a dataset that we will encode and store. The dataset [`jamescalam/ai-arxiv-chunked`](https://huggingface.co/datasets/jamescalam/ai-arxiv-chunked) contains scraped data from many popular ArXiv papers centred around LLMs. Including papers from Llama 2, GPTQ, and the GPT-4 technical paper.

In [2]:
%pip install pyarrow==20.0.0

  Using cached pyarrow-20.0.0-cp310-cp310-win_amd64.whl.metadata (3.4 kB)
Using cached pyarrow-20.0.0-cp310-cp310-win_amd64.whl (25.8 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 25.0.1
    Uninstalling pyarrow-25.0.1:
      Successfully uninstalled pyarrow-25.0.1
Note: you may need to restart the kernel to use updated packages.


In [2]:
import dill
import datasets

print("dill:", dill.__version__)
print("datasets:", datasets.__version__)
print("Has _dill:", hasattr(dill, "_dill"))

dill: 0.3.7
datasets: 2.14.5
Has _dill: True


In [3]:
from datasets import load_dataset

data = load_dataset("jamescalam/ai-arxiv-chunked", split="train")
data

Dataset({
    features: ['doi', 'chunk-id', 'chunk', 'id', 'title', 'summary', 'source', 'authors', 'categories', 'comment', 'journal_ref', 'primary_category', 'published', 'updated', 'references'],
    num_rows: 41584
})

We have 41.5K chunks, where each chunk is roughly the length of 1-2 paragraphs in length. Here is an example of a single record:

In [4]:
data[0]

{'doi': '1910.01108',
 'chunk-id': '0',
 'chunk': 'DistilBERT, a distilled version of BERT: smaller,\nfaster, cheaper and lighter\nVictor SANH, Lysandre DEBUT, Julien CHAUMOND, Thomas WOLF\nHugging Face\n{victor,lysandre,julien,thomas}@huggingface.co\nAbstract\nAs Transfer Learning from large-scale pre-trained models becomes more prevalent\nin Natural Language Processing (NLP), operating these large models in on-theedge and/or under constrained computational training or inference budgets remains\nchallenging. In this work, we propose a method to pre-train a smaller generalpurpose language representation model, called DistilBERT, which can then be ﬁnetuned with good performances on a wide range of tasks like its larger counterparts.\nWhile most prior work investigated the use of distillation for building task-speciﬁc\nmodels, we leverage knowledge distillation during the pre-training phase and show\nthat it is possible to reduce the size of a BERT model by 40%, while retaining 97%\nof i

Format the data into the format we need, this will contain `id`, `text` (which we will embed), and `metadata`. For this use-case we don't need metadata but it can be useful to include so that if needed in the future we can make use of metadata filtering.

In [5]:
data = data.map(lambda x: {
    "id": f'{x["id"]}-{x["chunk-id"]}',
    "text": x["chunk"],
    "metadata": {
        "title": x["title"],
        "url": x["source"],
        "primary_category": x["primary_category"],
        "published": x["published"],
        "updated": x["updated"],
        "text": x["chunk"],
    }
})
# drop uneeded columns
data = data.remove_columns([
    "title", "summary", "source",
    "authors", "categories", "comment",
    "journal_ref", "primary_category",
    "published", "updated", "references",
    "doi", "chunk-id",
    "chunk"
])
data

Dataset({
    features: ['id', 'text', 'metadata'],
    num_rows: 41584
})

We need to define an embedding model to create our embedding vectors for retrieval, for that we will be using OpenAI's `text-embedding-3-small`. There is some cost associated with this model, so be aware of that.

In [6]:
import os
import openai
import getpass  # platform.openai.com

# get API key from top-right dropdown on OpenAI website
openai.api_key = os.getenv("OPENAI_API_KEY") or getpass.getpass("Enter your openai API key:")

embed_model = "text-embedding-3-small"

Now we create our vector DB to store our vectors. For this we need to get a [free Pinecone API key](https://app.pinecone.io) — the API key can be found in the "API Keys" button found in the left navbar of the Pinecone dashboard.

In [7]:
from pinecone import Pinecone

# initialize connection to pinecone (get API key at app.pinecone.io)
api_key = os.getenv("PINECONE_API_KEY") or getpass.getpass("Enter your pinecone API key:")

# configure client
pc = Pinecone(api_key=api_key)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [8]:
from pinecone import ServerlessSpec

spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

Creating an index, we set `dimension` equal to the dimensionality of `text-embedding-3-small` (`1536`), and use a `metric` also compatible with it (this can be either `cosine` or `dotproduct`). We also pass our `spec` to index initialization.

In [9]:
import time

index_name = "rerankers"
existing_indexes = [
    index_info["name"] for index_info in pc.list_indexes()
]

# check if index already exists (it shouldn't if this is first time)
if index_name not in existing_indexes:
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=1536,  # dimensionality of text-embedding-3-small
        metric='dotproduct',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
time.sleep(1)
# view index stats
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}

Define embedding function with OpenAI:

In [29]:
def embed(batch: list[str]) -> list[list[float]]:
    for attempt in range(5):
        try:
            res = openai.embeddings.create(input=batch, model=embed_model)
            return [record.embedding for record in res.data]  # success: stop here
        except openai.RateLimitError:
            time.sleep(2 ** attempt)
            print("Retrying...")
    raise RuntimeError("Failed to create embeddings.")

We can see the index is currently empty with a `total_vector_count` of `0`. We can begin populating it with embeddings from OpenAI's `text-embedding-3-small` like so:

**⚠️ WARNING: this cell embeds all 41,584 chunks, which costs money and takes time. Run it once. The index keeps the vectors, so later sessions can skip this cell.**

The progress bar below is from my first run, before the retry loop in `embed()` was fixed. That run embedded every batch five times, so expect yours to be several times faster.

In [14]:
embed_model = "text-embedding-3-small"

In [15]:
from tqdm.auto import tqdm

batch_size = 100  # how many embeddings we create and insert at once

for i in tqdm(range(0, len(data), batch_size)):
    # find end of batch
    i_end = min(len(data), i+batch_size)
    # create batch
    batch = data[i:i_end]
    embeds = embed(batch["text"])
    to_upsert = list(zip(batch["id"], embeds, batch["metadata"]))
    # upsert to Pinecone
    index.upsert(vectors=to_upsert)

100%|██████████| 416/416 [1:04:49<00:00,  9.35s/it]


Now let's test retrieval _without_ Cohere's reranking model.

In [16]:
def get_docs(query: str, top_k: int) -> list[str]:
    # encode query
    xq = embed([query])[0]
    # search pinecone index
    res = index.query(vector=xq, top_k=top_k, include_metadata=True)
    # get doc text
    docs = {x["metadata"]['text']: i for i, x in enumerate(res["matches"])}
    return docs

In [17]:
query = "can you explain why we would want to do rlhf?"
docs = get_docs(query, top_k=25)
print("\n---\n".join(docs.keys()))

preferences and values which are diﬃcult to capture by hard- coded reward functions.
RLHF works by using a pre-trained LM to generate text, which i s then evaluated by humans by, for example,
ranking two model generations for the same prompt. This data is then collected to learn a reward model
that predicts a scalar reward given any generated text. The r eward captures human preferences when
judging model output. Finally, the LM is optimized against s uch reward model using RL policy gradient
algorithms like PPO ( Schulman et al. ,2017). RLHF can be applied directly on top of a general-purpose LM
pre-trained via self-supervised learning. However, for mo re complex tasks, the model’s generations may not
be good enough. In such cases, RLHF is typically applied afte r an initial supervised ﬁne-tuning phase using
a small number of expert demonstrations for the correspondi ng downstream task ( Ramamurthy et al. ,2022;
Ouyang et al. ,2022;Stiennon et al. ,2020).
A successful example of RLHF 

Good, but can we get better?

## Reranking Responses

We can easily get the responses we need when we include _many_ responses, but this doesn't work well with LLMs. The recall performance for LLMs [decreases as we add more into the context window](https://www.pinecone.io/blog/why-use-retrieval-instead-of-larger-context/) — we call this excessive filling of the context window _"context stuffing"_.

Fortunately reranking offers us a solution that helps us find those records that may not be within the top-3 results, and pull them into a smaller set of results to be given to the LLM.

We will use Cohere's rerank endpoint for this, to use it you will need a [Cohere API key](https://dashboard.cohere.com/api-keys). Once you have your key you use it to create authenticate your Cohere client like so:

In [18]:
import cohere

os.environ["COHERE_API_KEY"] = os.getenv("COHERE_API_KEY") or getpass.getpass()
# init client
co = cohere.Client(os.environ["COHERE_API_KEY"])

Now we can rerank our results with `co.rerank`. Let's try it with our earlier results.

In [19]:
rerank_docs = co.rerank(
    query=query, documents=docs.keys(), top_n=25, model="rerank-v4.0-fast"
)

This returns a list of `RerankResult` objects:

In [20]:
type(rerank_docs[0])

cohere.responses.rerank.RerankResult

We access the text content of the docs like so:

In [21]:
rerank_docs[0].document["text"]

'preferences and values which are diﬃcult to capture by hard- coded reward functions.\nRLHF works by using a pre-trained LM to generate text, which i s then evaluated by humans by, for example,\nranking two model generations for the same prompt. This data is then collected to learn a reward model\nthat predicts a scalar reward given any generated text. The r eward captures human preferences when\njudging model output. Finally, the LM is optimized against s uch reward model using RL policy gradient\nalgorithms like PPO ( Schulman et al. ,2017). RLHF can be applied directly on top of a general-purpose LM\npre-trained via self-supervised learning. However, for mo re complex tasks, the model’s generations may not\nbe good enough. In such cases, RLHF is typically applied afte r an initial supervised ﬁne-tuning phase using\na small number of expert demonstrations for the correspondi ng downstream task ( Ramamurthy et al. ,2022;\nOuyang et al. ,2022;Stiennon et al. ,2020).\nA successful examp

The reordered results look like so:

In [22]:
[docs[doc.document["text"]] for doc in rerank_docs]

[0,
 4,
 10,
 11,
 2,
 18,
 16,
 15,
 20,
 23,
 8,
 24,
 14,
 3,
 12,
 13,
 21,
 19,
 6,
 1,
 22,
 9,
 7,
 17,
 5]

Let's write a function to allow us to more easily compare the original results vs. reranked results.

In [25]:
def compare(query: str, top_k: int, top_n: int):
    # first get vec search results
    docs = get_docs(query, top_k=top_k)
    i2doc = {docs[doc]: doc for doc in docs.keys()}
    # rerank
    rerank_docs = co.rerank(
        query=query, documents=docs.keys(), top_n=top_n, model="rerank-v4.0-fast"
    )
    original_docs = []
    reranked_docs = []
    # compare order change
    for i, doc in enumerate(rerank_docs):
        rerank_i = docs[doc.document["text"]]
        print(str(i)+"\t->\t"+str(rerank_i))
        if i != rerank_i:
            reranked_docs.append(f"[{rerank_i}]\n"+doc.document["text"])
            original_docs.append(f"[{i}]\n"+i2doc[i])
    for orig, rerank in zip(original_docs, reranked_docs):
        print("ORIGINAL:\n"+orig+"\n\nRERANKED:\n"+rerank+"\n\n---\n")

Beginning with our `"can you explain why we would want to do rlhf?"` query, let's take a look at the top-3 results with / without reranking:

In [26]:
compare(query, 25, 3)

0	->	0
1	->	4
2	->	10
ORIGINAL:
[1]
handwritten by human annotators, suggesting that we could reprioritize and devote more annotation eﬀort
to preference-based annotation for RLHF.
Fine-Tuning Details. For supervised ﬁne-tuning, we use a cosine learning rate schedule with an initial
learning rate of 210 5, a weight decay of 0.1, a batch size of 64, and a sequence length of 4096 tokens.
For the ﬁne-tuning process, each sample consists of a prompt and an answer. To ensure the model sequence
lengthisproperlyﬁlled,weconcatenateallthepromptsandanswersfromthetrainingset. Aspecialtokenis
utilizedtoseparatethepromptandanswersegments. Weutilizeanautoregressiveobjectiveandzero-out
the loss on tokens from the user prompt, so as a result, we backpropagate only on answer tokens. Finally, we
ﬁne-tune the model for 2 epochs.
3.2 Reinforcement Learning with Human Feedback (RLHF)
RLHFisamodeltrainingprocedurethatisappliedtoaﬁne-tunedlanguagemodeltofurther alignmodel
behavior with human preferences and

Both results from reranking provide many more reasons as to why we would want to use RLHF than the original records. Let's try another query:

In [27]:
compare("what is red teaming?", top_k=25, top_n=3)

0	->	22
1	->	0
2	->	8
ORIGINAL:
[0]
including limitations and risks that might be exploited by m alicious actors. Further, existing
red teaming approaches are insufﬁcient for addressing thes e concerns in the AI context.
In order for AI developers to make veriﬁable claims about the ir AI systems being safe or secure, they need
processes for surfacing and addressing potential safety an d security risks. Practices such as red teaming
exercises help organizations to discover their own limitat ions and vulnerabilities as well as those of the
AI systems they develop, and to approach them holistically , in a way that takes into account the larger
environment in which they are operating.23
A red team exercise is a structured effort to ﬁnd ﬂaws and vuln erabilities in a plan, organization, or
technical system, often performed by dedicated "red teams" that seek to adopt an attacker’s mindset
and methods. In domains such as computer security , red teams are routinely tasked with emulating
attack

Again, the results provide more relevant responses when using reranking rather than the original search.

## Measuring the Reranker

Two queries compared by eye show that reranking *can* help. To know **how much** it helps, we need a test set and metrics. We will:

1. build a small test set of questions, where each question has one known answer chunk;
2. run every question through both pipelines: vector search only, and vector search + rerank;
3. compare where the answer chunk lands, and how long each step takes.

This section uses `data`, `index`, `embed()` and `co` from the cells above. It does **not** need the embed-and-upsert loop again: the index already holds the vectors.

In [28]:
import json
import math
import random
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

EVAL_SIZE = 50                 # number of test questions
EVAL_SEED = 7                  # same seed = same chunks on every run
EVAL_TOP_K = 25                # stage 1: how many chunks vector search returns
RERANK_MODEL = "rerank-v4.0-fast"
RERANK_PAUSE_S = 6.5           # Cohere trial keys allow 10 rerank calls a minute; use 0 with a production key

# the first chat model in this list that your OpenAI key can use writes the test questions
QUESTION_MODELS = ["gpt-6-luna", "gpt-5-mini", "gpt-4.1-mini", "gpt-4o-mini"]

EVAL_FILE = Path("rerank_eval_set.json")         # the test set, saved so every run uses the same questions
RESULTS_FILE = Path("rerank_eval_results.json")  # raw results of the last run

### 1. Build the test set

We pick random chunks and ask an LLM to write one question that each chunk answers. That chunk's `id` is the right answer (the *gold* chunk) for its question.

The questions are saved to `rerank_eval_set.json`. Later runs load that file instead of writing new questions, so results stay comparable when we change a model or a setting.

In [31]:
QUESTION_PROMPT = """Below is a passage from a research paper.
Write ONE specific question that this passage answers.

Rules:
- Someone who has never seen the passage must be able to understand the question.
- Name the method, model or dataset the passage is about, so the question fits only this topic.
- Do not write "the passage", "this paper" or "the authors".
- If the passage is a reference list, a table of numbers, or has no clear fact to ask about, reply with only: SKIP

Passage:
{passage}"""


def ask_llm(prompt: str) -> str:
    """One chat call, with the first model in QUESTION_MODELS that this key can use."""
    waits = 0
    while QUESTION_MODELS:
        try:
            res = openai.chat.completions.create(
                model=QUESTION_MODELS[0],
                messages=[{"role": "user", "content": prompt}],
            )
            return (res.choices[0].message.content or "").strip()
        except openai.RateLimitError:
            waits += 1
            if waits > 5:
                raise
            print("OpenAI rate limit, waiting 10 s...")
            time.sleep(10)
        except (openai.NotFoundError, openai.PermissionDeniedError, openai.BadRequestError) as e:
            print(f"Can't use {QUESTION_MODELS[0]} ({type(e).__name__}), trying the next model.")
            QUESTION_MODELS.pop(0)
    raise RuntimeError("No chat model worked. Put one your key can use in QUESTION_MODELS.")


def build_eval_set() -> list[dict]:
    """Pick random chunks and write one question for each. The chunk is the right answer."""
    rng = random.Random(EVAL_SEED)
    # take extra candidates, because some chunks get skipped
    candidates = rng.sample(range(len(data)), min(len(data), EVAL_SIZE * 3))
    eval_set = []
    for i in tqdm(candidates):
        if len(eval_set) == EVAL_SIZE:
            break
        chunk = data[i]
        if len(chunk["text"]) < 500:  # too short to hold a clear fact
            continue
        question = ask_llm(QUESTION_PROMPT.format(passage=chunk["text"]))
        if not question or question.upper().startswith("SKIP"):
            continue
        eval_set.append({"question": question, "gold_id": chunk["id"], "gold_text": chunk["text"]})
    print("Questions written by", QUESTION_MODELS[0])
    return eval_set


if EVAL_FILE.exists():
    eval_set = json.loads(EVAL_FILE.read_text(encoding="utf-8"))
    print(f"Loaded {len(eval_set)} questions from {EVAL_FILE}")
else:
    eval_set = build_eval_set()
    EVAL_FILE.write_text(json.dumps(eval_set, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Wrote {len(eval_set)} questions to {EVAL_FILE}")

for row in eval_set[:5]:
    print("-", row["question"])

Loaded 50 questions from rerank_eval_set.json
- What natural-language request corresponds to the SQL query in the sql2text example from the UnifiedSKG benchmark?
- For the machine-learning tasks tested, what range of critical batch sizes did the simple gradient noise scale predict to within an order of magnitude?
- Why does the socialbot A/B testing evaluation discard scores from returning users?
- For an LSTM recurrent neural network trained on text, what quantity do the authors propose measuring as a computationally efficient diagnostic of its potential for improvement?
- What approximation factor do Lei et al.’s paraphrase-based adversarial attack authors prove for their greedy algorithm on CNN and RNN text classifiers?


Read a few questions in `rerank_eval_set.json` next to their `gold_text` before trusting any number. A bad test set gives confident, wrong results.

### 2. Run both pipelines

For each question we:

1. embed it and get the top 25 chunks from Pinecone (stage 1);
2. rerank all 25 with Cohere (stage 2);
3. record the position of the gold chunk before and after, and the time each step took.

We rerank all 25, not only the top 3, so one call tells us the result for every cutoff. With a trial key this cell takes about 6 minutes because of the pause between calls.

In [32]:
def rerank_timed(query: str, texts: list[str]):
    """Rerank all the texts. Returns the result and the seconds the successful call took.

    If Cohere answers "too many requests" (HTTP 429), wait and try again.
    """
    for attempt in range(6):
        start = time.perf_counter()
        try:
            reranked = co.rerank(query=query, documents=texts, top_n=len(texts), model=RERANK_MODEL)
            return reranked, time.perf_counter() - start
        except cohere.CohereAPIError as e:
            if e.http_status != 429:
                raise
            print("Cohere rate limit, waiting 20 s...")
            time.sleep(20)
    raise RuntimeError("Cohere is still rate limiting after 6 tries.")


def evaluate_question(row: dict) -> dict:
    """Run one question through both stages.

    Positions start at 1. None means the gold chunk was not in the top EVAL_TOP_K.
    """
    t0 = time.perf_counter()
    xq = embed([row["question"]])[0]
    t1 = time.perf_counter()
    matches = index.query(vector=xq, top_k=EVAL_TOP_K, include_metadata=True)["matches"]
    t2 = time.perf_counter()

    ids = [m["id"] for m in matches]                 # order from vector search
    texts = [m["metadata"]["text"] for m in matches]
    reranked, rerank_s = rerank_timed(row["question"], texts)
    reranked_ids = [ids[r.index] for r in reranked]  # order after reranking

    def position(id_list):
        return id_list.index(row["gold_id"]) + 1 if row["gold_id"] in id_list else None

    return {
        "question": row["question"],
        "gold_id": row["gold_id"],
        "vector_pos": position(ids),
        "rerank_pos": position(reranked_ids),
        "embed_ms": (t1 - t0) * 1000,
        "search_ms": (t2 - t1) * 1000,
        "rerank_ms": rerank_s * 1000,
    }


results = []
for row in tqdm(eval_set):
    results.append(evaluate_question(row))
    time.sleep(RERANK_PAUSE_S)

RESULTS_FILE.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved {len(results)} results to {RESULTS_FILE}")

100%|██████████| 50/50 [06:35<00:00,  7.91s/it]


Saved 50 results to rerank_eval_results.json


### 3. Metrics

- **hit@k**: the share of questions whose gold chunk is in the top `k`. With one gold chunk per question, this is the same as recall@k. **hit@3 is the one to watch**, because the top 3 is what we would pass to the LLM.
- **MRR** (mean reciprocal rank): `1 / position` of the gold chunk, averaged over all questions. A chunk that was not retrieved counts as 0.
- **nDCG@k**: like hit@k, but a chunk scores more the higher it is: `1 / log2(position + 1)`.

Reranking only reorders what stage 1 returned, so it cannot change hit@25. That number is the ceiling for the reranker.

In [33]:
def hit_at(positions, k):
    return sum(p is not None and p <= k for p in positions) / len(positions)


def mrr(positions):
    return sum(1 / p for p in positions if p is not None) / len(positions)


def ndcg_at(positions, k):
    # one gold chunk per question, so the best possible score for a question is 1
    return sum(1 / math.log2(p + 1) for p in positions if p is not None and p <= k) / len(positions)


vector_pos = [r["vector_pos"] for r in results]
rerank_pos = [r["rerank_pos"] for r in results]

rows = [(f"hit@{k}", hit_at(vector_pos, k), hit_at(rerank_pos, k)) for k in (1, 3, 5, 10)]
rows.append(("MRR", mrr(vector_pos), mrr(rerank_pos)))
rows += [(f"nDCG@{k}", ndcg_at(vector_pos, k), ndcg_at(rerank_pos, k)) for k in (3, 10)]

summary = pd.DataFrame(rows, columns=["metric", "vector only", "vector + rerank"]).set_index("metric")
summary["change"] = summary["vector + rerank"] - summary["vector only"]

print(f"{len(results)} questions, top_k = {EVAL_TOP_K}, reranker = {RERANK_MODEL}")
print(f"Gold chunk found by stage 1 (hit@{EVAL_TOP_K}, the ceiling): {hit_at(vector_pos, EVAL_TOP_K):.0%}")
summary.round(3)

50 questions, top_k = 25, reranker = rerank-v4.0-fast
Gold chunk found by stage 1 (hit@25, the ceiling): 94%


,vector only,vector + rerank,change
metric,,,
hit@1,0.600,0.780,0.180
hit@3,0.820,0.920,0.100
hit@5,0.840,0.920,0.080
hit@10,0.920,0.940,0.020
MRR,0.718,0.849,0.131
nDCG@3,0.734,0.866,0.132
nDCG@10,0.766,0.872,0.106


How often did reranking move the gold chunk up or down, and what does it cost in time?

In [34]:
def what_happened(r: dict) -> str:
    if r["vector_pos"] is None:
        return "missed by stage 1"
    if r["rerank_pos"] < r["vector_pos"]:
        return "moved up"
    if r["rerank_pos"] > r["vector_pos"]:
        return "moved down"
    return "same position"


print(pd.Series([what_happened(r) for r in results]).value_counts().to_string())

# time per step, in milliseconds
latency = pd.DataFrame(results)[["embed_ms", "search_ms", "rerank_ms"]]
latency.describe(percentiles=[0.5, 0.95]).loc[["50%", "95%", "mean"]].round(0)

same position        29
moved up             15
missed by stage 1     3
moved down            3


,embed_ms,search_ms,rerank_ms
50%,412.0,140.0,455.0
95%,796.0,266.0,2743.0
mean,485.0,172.0,743.0


Numbers hide the interesting cases, so look at them: the biggest improvements, the questions where reranking made things worse, and the ones stage 1 missed.

In [35]:
df = pd.DataFrame(results)
df[["vector_pos", "rerank_pos"]] = df[["vector_pos", "rerank_pos"]].astype("Int64")
df["moved_up_by"] = df["vector_pos"] - df["rerank_pos"]  # positive = the reranker moved the gold chunk up
columns = ["question", "vector_pos", "rerank_pos", "moved_up_by"]
pd.set_option("display.max_colwidth", 120)

print("Biggest improvements:")
display(df.sort_values("moved_up_by", ascending=False).head(5)[columns])

print("Reranking made these worse:")
display(df[df["moved_up_by"] < 0].sort_values("moved_up_by")[columns])

print(f"Gold chunk not in the top {EVAL_TOP_K} (reranking cannot fix these):")
display(df[df["vector_pos"].isna()][["question"]])

Biggest improvements:


,question,vector_pos,rerank_pos,moved_up_by
0,What natural-language request corresponds to the SQL query in the sql2text example from the UnifiedSKG benchmark?,19,1,18
8,What three qualities define an aligned AI assistant under Anthropic’s HHH alignment criteria?,10,1,9
46,What results does Table 9 report for models trained on MNLI with neutral and contradiction merged into a single non-...,10,1,9
30,What is the main goal of subword-unit neural machine translation models for handling rare and unseen words?,10,3,7
47,What does the GOAL MATRIX observation encode for the agents described in *Open-Ended Learning Leads to Generally Cap...,7,2,5


Reranking made these worse:


,question,vector_pos,rerank_pos,moved_up_by
22,In the Hessian top-subspace overlap analysis of a two-layer fully connected network on MNIST and ResNet-18 on CIFAR-...,1,9,-8
29,"According to the GHG Protocol breakdown of Google’s and Facebook’s data-center carbon footprints, what types of acti...",1,2,-1
32,What output does the letter-position concatenation procedure produce for “Alan Mathison Turing”?,1,2,-1


Gold chunk not in the top 25 (reranking cannot fix these):


,question
5,How do control codes influence text generation when the prompt is identical?
38,What top-1 and top-5 accuracy values are listed for SENet?
49,"How does the amount of training data needed scale with language model size, according to the reported findings?"


### Reading the results

- **Small test set, noisy numbers.** With 50 questions, one question is 2 points. Trust differences of 10 points or more, and treat smaller ones as a hint.
- **The questions share words with their chunks,** because an LLM wrote them from the chunks. That flatters vector search. Real user questions are vaguer, so vector search alone will usually score lower on them than it does here.
- **One gold chunk per question.** Another chunk may answer the same question and count as a miss, so these numbers are a lower bound.
- **Reranking is not free.** Compare `rerank_ms` with `embed_ms` and `search_ms` above. That time is added to every query.
- **To test a change** (another rerank model, or `EVAL_TOP_K = 50`), keep `rerank_eval_set.json` and run this section again from step 2. Same questions, so the numbers are comparable.

Don't forget to delete your index when you're done to save resources!

In [ ]:
#pc.delete_index(index_name)

---